# 07 Tool layers side by side: schema reconciliation, cross-implementation checks, coverage

**Date:** 2026-10-22
**Author:** Xincheng (George) Ding
**Needs network** for the Cypher half when `IATI_LIVE_EVAL=1`; the pandas half runs offline on the pinned snapshot.

**Why.** Two tool layers now exist: the 16 benchmark-shaped Cypher templates in `fa26/src/tools/` (notebook 06) and
Alex's four user-facing retrieval tools in `fa26/retrieval/` (branch `alex`, commit 77e04b1). They were written
against **different graph schemas** (Alex's loader casts dates and values and uses `PROVIDES`; the Aura graph loaded by
04 v2 keeps strings, uses `PROVIDED_BY` and carries `source_id`), so his tools cannot run on the shared Aura instance
yet. Before choosing or merging, this notebook establishes facts:

- **A. Schema reconciliation.** What the Aura graph actually holds (introspected) versus what Alex's Cypher expects
  (read from his `schema_prototype.py`), and the smallest unified schema v0.2 that serves both.
- **B. Cross-implementation agreement.** The questions both layers answer, computed three ways: Cypher template on
  Aura, an independent pandas/Decimal implementation on the pinned CSVs (`src/tools/crosscheck.py`), and the numbers
  Alex published in `retrieval/findings.md`. Agreement here is the same kind of evidence as the benchmark's SQL baseline.
- **C. Entity resolution effect.** Shared partners between Oxfam GB and WFP before and after identifier canonicalisation.
- **D. Coverage.** Which toolset answers which question type, and which can cite row-level evidence.

Outputs: `fa26/data/interim/07_crosscheck_<run>.csv`, `07_schema_diff_<run>.csv`.

In [1]:
from pathlib import Path
import os, sys, re, json, csv
from decimal import Decimal
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fa26" / "benchmark" / "manifest.json").exists())
FA26 = ROOT / "fa26"
sys.path.insert(0, str(FA26 / "benchmark")); sys.path.insert(0, str(FA26 / "src"))
from common import verify_snapshot
from text2cypher_eval import guard_query
from tools import TOOLS, run_tool, OrgResolver
from tools import crosscheck as cc
SNAPSHOT_ID = verify_snapshot()["snapshot_id"]
SNAP = FA26 / "benchmark" / "snapshot"

import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv(FA26 / ".env", override=True)
DRY_RUN = os.environ.get("IATI_LIVE_EVAL", "0") != "1"
RUN_ID = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M")
INTERIM = FA26 / "data" / "interim"; INTERIM.mkdir(parents=True, exist_ok=True)
ALEX = FA26 / "retrieval"
print("run", RUN_ID, "| dry run:", DRY_RUN, "| Alex's retrieval package present:", ALEX.exists())
if not ALEX.exists():
    print("To include it: git checkout origin/alex -- fa26/retrieval   (unchanged copy, credited to Alex)")

run 2026-10-22_2043 | dry run: False | Alex's retrieval package present: False
To include it: git checkout origin/alex -- fa26/retrieval   (unchanged copy, credited to Alex)


In [2]:
if DRY_RUN:
    def run_cypher(q, params=None): return []
    print("dry run: Cypher checks skipped, pandas checks run")
else:
    from neo4j import GraphDatabase, Query, READ_ACCESS
    driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))
    driver.verify_connectivity(); DB = os.environ.get("NEO4J_DATABASE", "neo4j")
    def run_cypher(q, params=None):
        guard_query(q)
        with driver.session(database=DB, default_access_mode=READ_ACCESS) as s:
            return [dict(r) for r in s.run(Query(q, timeout=60), **(params or {}))]
    print("Neo4j connected")
T = cc.load(SNAP)
print({k: len(v) for k, v in T.items() if not k.startswith("_") and len(v)})

Neo4j connected
{'nodes_activities': 505, 'nodes_budgets': 2999, 'nodes_countries': 13, 'nodes_locations': 82, 'nodes_organisations': 238, 'nodes_sdgs': 2, 'nodes_sectors': 17, 'nodes_transactions': 23423, 'rel_classified_as': 1238, 'rel_contributes_to': 82, 'rel_located_at': 82, 'rel_participates': 3457, 'rel_reports': 505, 'rel_targets_country': 505, 'rel_transaction_edges': 26830}


## A. Schema reconciliation: the Aura graph as loaded versus what Alex's Cypher expects

*Update 2026-11-05: Aura is now loaded by `fa26/retrieval/load_neo4j.py` (schema prototype-v0) and `src/tools/templates.py` was rewritten to it, so every row below should read as present/typed. The first run (2026-10-22) recorded the 04 v2 graph.*

In [3]:
# What Alex's tools and loader assume, read from fa26/retrieval/schema_prototype.py and tools.py (branch alex, d2f529d).
ALEX_EXPECTS = [
    ("Transaction.value",  "float (cast at load)",          "t.value summed directly"),
    ("Transaction.date",   "Date (cast at load)",           "t.date.year, min/max(t.date)"),
    ("Transaction.type_name", "derived label, e.g. 'Incoming Funds'", "WHERE t.type_name IN $types"),
    ("Activity.start_date / end_date", "derived Date = actual else planned", "a.start_date.year filters"),
    ("Activity.status_name", "derived label",               "returned to the user"),
    ("Activity.hotspot_countries", "list (split on |)",     "returned to the user"),
    ("Organisation.publisher", "slug on reporting orgs",    "_publisher_ref lookup"),
    ("PARTICIPATES_IN.role_name", "derived label",          "shared_partners roles"),
    ("(Organisation)-[:PROVIDES]->(Transaction)", "relationship direction", "top_funders, shared_partners"),
    ("rel_participates duplicates", "collapsed (210 exact duplicate rows -> one edge)", "counts: 58,911 relationships"),
]
# What the Aura graph holds. valueType() needs Neo4j 5.13+; Aura Free is current.
def aura_type(expr, label):
    if DRY_RUN: return "n/a (dry run)"
    rows = run_cypher(f"MATCH (n:{label}) WHERE {expr} IS NOT NULL RETURN valueType({expr}) AS t LIMIT 1")
    return rows[0]["t"] if rows else "missing"
def aura_has_prop(label, prop):
    if DRY_RUN: return "n/a (dry run)"
    rows = run_cypher(f"MATCH (n:{label}) WHERE n.{prop} IS NOT NULL RETURN count(n) AS c")
    return "present" if rows and rows[0]["c"] else "missing"
def aura_rel(rtype):
    if DRY_RUN: return "n/a (dry run)"
    rows = run_cypher(f"MATCH ()-[r:{rtype}]->() RETURN count(r) AS c")
    return f"{rows[0]['c']} edges" if rows and rows[0]["c"] else "missing"
AURA_HAS = {
    "Transaction.value": aura_type("n.value", "Transaction"),
    "Transaction.date": aura_type("n.date", "Transaction"),
    "Transaction.type_name": aura_has_prop("Transaction", "type_name"),
    "Activity.start_date / end_date": aura_has_prop("Activity", "start_date"),
    "Activity.status_name": aura_has_prop("Activity", "status_name"),
    "Activity.hotspot_countries": aura_type("n.hotspot_countries", "Activity"),
    "Organisation.publisher": aura_has_prop("Organisation", "publisher"),
    "PARTICIPATES_IN.role_name": "n/a (dry run)" if DRY_RUN else ("present" if run_cypher("MATCH ()-[r:PARTICIPATES_IN]->() WHERE r.role_name IS NOT NULL RETURN count(r) AS c")[0]["c"] else "missing"),
    "(Organisation)-[:PROVIDES]->(Transaction)": aura_rel("PROVIDES") + "; PROVIDED_BY: " + aura_rel("PROVIDED_BY"),
    "rel_participates duplicates": aura_rel("PARTICIPATES_IN") + " (3,247 = collapsed; 3,457 = one edge per row)",
}
schema_diff = pd.DataFrame([{"item": i, "alex_expects": e, "used_by": u, "aura_has": AURA_HAS[i]} for i, e, u in ALEX_EXPECTS])
schema_diff.to_csv(INTERIM / f"07_schema_diff_{RUN_ID}.csv", index=False)
pd.set_option("display.max_colwidth", 80); schema_diff

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `type_name` does not exist in database `9391175d`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=31, offset=30>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 30, 'line': 1, 'column': 31}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n:Transaction) WHERE n.type_name IS NOT NULL RETURN count(n) AS c'
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `start_date` does not exist in database `9391175d`. Verify that the spelling is correct.', 

,item,alex_expects,used_by,aura_04v2_has
0,Transaction.value,float (cast at load),t.value summed directly,STRING NOT NULL (value_num is FLOAT NOT NULL)
1,Transaction.date,Date (cast at load),"t.date.year, min/max(t.date)",STRING NOT NULL
2,Transaction.type_name,"derived label, e.g. 'Incoming Funds'",WHERE t.type_name IN $types,missing (type_label present from 04: present)
3,Activity.start_date / end_date,derived Date = actual else planned,a.start_date.year filters,missing
4,Activity.status_name,derived label,returned to the user,missing
5,Activity.hotspot_countries,list (split on |),returned to the user,STRING NOT NULL
6,Organisation.publisher,slug on reporting orgs,_publisher_ref lookup,missing
7,PARTICIPATES_IN.role_name,derived label,shared_partners roles,missing
8,(Organisation)-[:PROVIDES]->(Transaction),relationship direction,"top_funders, shared_partners",missing; PROVIDED_BY: 17570 edges
9,rel_participates duplicates,collapsed (210 exact duplicate rows -> one edge),"counts: 58,911 relationships",kept as separate edges keyed by source_id: 3457 edges


**Superseded 2026-11-05.** The team standard is Alex's loader and schema (`retrieval/schema_prototype.py`); notebook 04 is retired and the templates were rewritten to that schema (commit 6c1e184). The proposal below is kept for the record.

**Proposed unified schema v0.2** (as written 2026-10-22):

- keep `source_id` on every node and relationship, and keep one edge per source row (benchmark evidence needs it);
  expose a deduplicated view through `DISTINCT` in queries rather than dropping rows at load;
- keep the exported strings (`value`, `date`) and add typed twins: `value_num` (already there), `date_d` (Date),
  `start_date` / `end_date` (Date, actual else planned); templates keep string comparison, Alex's tools use the typed twins;
- add the label properties Alex's tools filter on: `type_name`, `status_name`, `role_name`, `Organisation.publisher`;
- add `PROVIDES` as a second spelling of `PROVIDED_BY` (reverse direction, same `source_id`), or change Alex's four
  queries to `<-[:PROVIDED_BY]-`; the second is a four-line edit and avoids 17,570 duplicate edges.

## B. Cross-implementation agreement: Cypher template on Aura vs pandas/Decimal on the pinned CSVs vs Alex's published numbers

In [4]:
def dec(v): return Decimal(str(v)).quantize(Decimal("0.01"))
checks = []
def compare(name, cypher_rows, pandas_map, alex_note=""):
    cy = {str(r["key"]): dec(r["value"]) for r in cypher_rows} if cypher_rows else {}
    keys = sorted(set(cy) | set(pandas_map), key=lambda k: -pandas_map.get(k, Decimal(0)))
    for k in keys:
        a, b = cy.get(k), dec(pandas_map[k]) if k in pandas_map else None
        checks.append({"check": name, "key": k, "cypher_template": None if a is None else str(a), "pandas_csv": None if b is None else str(b),
                       "agree": (a is not None and b is not None and abs(a - b) <= Decimal("0.01")) if not DRY_RUN else None, "alex_findings": alex_note})

# B1. Top funders of Oxfam GB's activities, incoming funds, GBP (template top_funders vs pandas)
rows = [] if DRY_RUN else run_tool(run_cypher, "top_funders", {"publisher": "oxfamgb", "type_code": "1", "currency": "GBP", "limit": 5})["rows"]
compare("B1 top_funders oxfamgb type1 GBP", rows, cc.top_funders(T, "oxfamgb", "1", "GBP", 5), "WFP is the largest GBP funder of Oxfam GB (findings.md, demo claim)")

# B2. Shared partners Oxfam GB x WFP, raw identifiers (template shared_partners vs pandas)
rows = [] if DRY_RUN else run_tool(run_cypher, "shared_partners", {"publisher_a": "oxfamgb", "publisher_b": "wfp"})["rows"]
pm = {k: Decimal(v) for k, v in cc.shared_partners(T, "oxfamgb", "wfp").items()}
if not pm and not rows: checks.append({"check": "B2 shared_partners oxfamgb x wfp (raw refs)", "key": "(none)", "cypher_template": "0 rows", "pandas_csv": "0 rows", "agree": None if DRY_RUN else True, "alex_findings": "zero shared orgs on raw refs; same funders carry different IDs per publisher"})
else: compare("B2 shared_partners oxfamgb x wfp (raw refs)", rows, pm, "findings.md: zero shared orgs on raw refs")

# B3. WFP expenditure (type 4) USD per year (template funding_over_time vs pandas)
rows = [] if DRY_RUN else run_tool(run_cypher, "funding_over_time", {"publisher": "wfp", "type_code": "4", "currency": "USD"})["rows"]
compare("B3 funding_over_time wfp type4 USD", rows, cc.funding_over_time(T, "wfp", "4", "USD"), "WFP transactions begin 2021; type 2 == type 4 to the dollar")

# B4. WFP commitment/expenditure mirror (pandas only; Alex: 18,037,403,740 positive, 20,011,330 negative, 2,756 rows each)
m = cc.wfp_mirror(T)
checks.append({"check": "B4 WFP type2 == type4 mirror", "key": "positive total", "cypher_template": None, "pandas_csv": f"type2 {m['2']['positive']} / type4 {m['4']['positive']} (n={m['2']['n']}/{m['4']['n']})",
               "agree": m["2"] == m["4"], "alex_findings": "18,037,403,740 positive, 2,756 rows each"})

# B5. Country-level top funders into Somalia (Alex's definition; ad-hoc Cypher here, not a template)
SOMALIA_CYPHER = '''
MATCH (a:Activity)-[:TARGETS]->(:Country {code: 'SO'}), (a)-[:HAS_TRANSACTION]->(t:Transaction {type: '1', currency: 'USD'})<-[:PROVIDES]-(o:Organisation),
      (rep:Organisation)-[:REPORTS]->(a)
WHERE o.ref <> rep.ref
RETURN o.ref + ' [USD]' AS key, sum(t.value) AS value, [] AS evidence ORDER BY value DESC LIMIT 5'''
rows = [] if DRY_RUN else run_cypher(SOMALIA_CYPHER.strip())
compare("B5 top funders into Somalia type1 USD excl self", rows, cc.country_top_funders(T, "SO", ("1",), "USD", True, 5),
        "findings.md app table: United States 1.20bn, Somalia 343m, Multilateral 135m, FCDO 59m")

# B6. Participation duplicates explain the three relationship counts
d = cc.participation_duplicates(T)
checks.append({"check": "B6 rel_participates exact duplicates", "key": "duplicates", "cypher_template": None, "pandas_csv": str(d),
               "agree": d == 210, "alex_findings": "210 duplicate rows collapse to one edge: 59,121 - 210 = 58,911"})
cross = pd.DataFrame(checks); cross.to_csv(INTERIM / f"07_crosscheck_{RUN_ID}.csv", index=False)
pd.set_option("display.max_colwidth", 60); cross

,check,key,cypher_template,pandas_csv,agree,alex_findings
0,B1 top_funders oxfamgb type1 GBP,name:UN World Food Programme (WFP) [GBP],203438175.00,203438175.00,True,"WFP is the largest GBP funder of Oxfam GB (findings.md, ..."
1,B1 top_funders oxfamgb type1 GBP,XM-DAC-12-1 [GBP],42241092.00,42241092.00,True,"WFP is the largest GBP funder of Oxfam GB (findings.md, ..."
2,B1 top_funders oxfamgb type1 GBP,name:EU - ECHO (EC Humanitarian Aid Dept) [GBP],38091027.00,38091027.00,True,"WFP is the largest GBP funder of Oxfam GB (findings.md, ..."
3,B1 top_funders oxfamgb type1 GBP,DK-CVR-88136411 [GBP],23534246.00,23534246.00,True,"WFP is the largest GBP funder of Oxfam GB (findings.md, ..."
4,B1 top_funders oxfamgb type1 GBP,name:Oxfam Germany [GBP],22664307.00,22664307.00,True,"WFP is the largest GBP funder of Oxfam GB (findings.md, ..."
5,B2 shared_partners oxfamgb x wfp (raw refs),(none),0 rows,0 rows,True,zero shared orgs on raw refs; same funders carry differe...
6,B3 funding_over_time wfp type4 USD,2022,5263780779.60,5263780779.60,True,WFP transactions begin 2021; type 2 == type 4 to the dollar
7,B3 funding_over_time wfp type4 USD,2023,4307449043.44,4307449043.44,True,WFP transactions begin 2021; type 2 == type 4 to the dollar
8,B3 funding_over_time wfp type4 USD,2024,3573521569.95,3573521569.95,True,WFP transactions begin 2021; type 2 == type 4 to the dollar
9,B3 funding_over_time wfp type4 USD,2025,3288849657.77,3288849657.77,True,WFP transactions begin 2021; type 2 == type 4 to the dollar


In [5]:
if not DRY_RUN:
    print("agreement:", cross.agree.value_counts(dropna=False).to_dict())
    bad = cross[cross.agree == False]
    print("disagreements:" if len(bad) else "no disagreements"); display(bad) if len(bad) else None

agreement: {True: 18}
no disagreements


## C. Entity resolution effect: shared partners Oxfam GB x WFP, raw identifiers versus canonical groups

In [6]:
from collections import defaultdict
resolver = OrgResolver.from_csv(SNAP / "nodes_organisations.csv")
canon = {ref: g["group"] for g in resolver.duplicate_groups() if g["group"] != "OXFAM_FAMILY" for ref in g["refs"]}
tx = {x["tx_id"]: x for x in T["nodes_transactions"]}
pubs, acts = defaultdict(set), defaultdict(set)
def add(ref, a):
    c = canon.get(ref, ref); pubs[c].add(T["_act"][a]["publisher"]); acts[c].add(a)
for x in T["rel_participates"]: add(x["org_ref"], x["iati_identifier"])
for e in T["rel_transaction_edges"]:
    if e["edge"] == "PROVIDED_BY": add(e["org_ref"], tx[e["tx_id"]]["iati_identifier"])
publishers_self = {"GB-CHC-202918", "XM-DAC-41140", "WFP", "US-EIN-23-7069110"}
raw = {c: len(acts[c]) for c in acts if {"oxfamgb", "wfp"} <= pubs[c] and c not in canon.values() and c not in publishers_self}
merged = {c: len(acts[c]) for c in acts if {"oxfamgb", "wfp"} <= pubs[c] and c in set(canon.values()) and c not in publishers_self}
print(f"shared organisations Oxfam GB x WFP (participation + funding edges): raw refs = {len(raw)}, after canonical groups = {len(merged)}")
print(pd.Series(merged).sort_values(ascending=False).to_string())
print("\nAlex reports 12 shared funders with his SAME_AS/AGENCY_OF tables (slide); ours finds", len(merged), "with 11 alias groups: merge the two alias tables.")

shared organisations Oxfam GB x WFP (participation + funding edges): raw refs = 0, after canonical groups = 7
EU            101
GERMANY        97
US_GOV         81
UK_GOV         69
FRANCE         64
UNICEF         57
WORLD_BANK     25

Alex reports 12 shared funders with his SAME_AS/AGENCY_OF tables (slide); ours finds 7 with 11 alias groups: merge the two alias tables.


## D. Coverage: which toolset answers which question type, and who can cite row-level evidence

In [7]:
coverage = pd.DataFrame([
 # question type                               templates(06)  alex retrieval   evidence(source_id)   note
 ("Count activities per publisher / country / basis / sdg / location", "yes", "activities() returns the list; count by caller", "templates yes; alex activity_ids only", "benchmark Q01-03, Q06, Q15-16"),
 ("Participation roles, multi-publisher participants",       "yes", "shared_partners() partially", "templates yes", "Q04, Q05"),
 ("Sector count for one publisher",                           "yes", "no", "templates yes", "Q07"),
 ("Latest update date per publisher",                         "yes", "no", "templates yes", "Q09"),
 ("Net transaction total (publisher, type, currency, period)","yes", "funding_over_time() per period", "templates yes", "Q10-Q12; alex splits positive/negative"),
 ("Negative transactions per publisher",                      "yes", "n_negative inside other tools", "templates yes", "Q13"),
 ("Budget totals",                                            "yes", "no", "templates yes", "Q14"),
 ("Top funders INTO A COUNTRY (user question)",               "no (publisher-level only)", "top_funders(country, ...) yes, excludes self, canonical IDs", "neither; alex activity_ids", "Alex's demo"),
 ("Shared partners across publishers (user question)",        "yes (raw refs)", "yes, with canonical IDs and roles", "templates yes", "needs merged alias table"),
 ("Funding over time by quarter/month",                       "year only", "yes (year/quarter/month)", "templates yes", ""),
 ("Natural-language organisation / country names",            "via resolver (11 groups)", "via resolve_entity (rapidfuzz + curated tables)", "", "merge"),
 ("Coverage note on every answer",                            "static per template", "conditional, from findings", "", "adopt Alex's"),
 ("Abstain / clarify handling",                               "router policy (06)", "no", "", "router stays in 06"),
], columns=["question type", "templates (src/tools)", "alex (fa26/retrieval)", "row-level evidence", "note"])
pd.set_option("display.max_colwidth", 70); coverage

,question type,templates (src/tools),alex (fa26/retrieval),row-level evidence,note
0,Count activities per publisher / country / basis / sdg / location,yes,activities() returns the list; count by caller,templates yes; alex activity_ids only,"benchmark Q01-03, Q06, Q15-16"
1,"Participation roles, multi-publisher participants",yes,shared_partners() partially,templates yes,"Q04, Q05"
2,Sector count for one publisher,yes,no,templates yes,Q07
3,Latest update date per publisher,yes,no,templates yes,Q09
4,"Net transaction total (publisher, type, currency, period)",yes,funding_over_time() per period,templates yes,Q10-Q12; alex splits positive/negative
5,Negative transactions per publisher,yes,n_negative inside other tools,templates yes,Q13
6,Budget totals,yes,no,templates yes,Q14
7,Top funders INTO A COUNTRY (user question),no (publisher-level only),"top_funders(country, ...) yes, excludes self, canonical IDs",neither; alex activity_ids,Alex's demo
8,Shared partners across publishers (user question),yes (raw refs),"yes, with canonical IDs and roles",templates yes,needs merged alias table
9,Funding over time by quarter/month,year only,yes (year/quarter/month),templates yes,


## E. What this establishes (facts first, decisions second)

*Update 2026-11-05: decisions 1 to 3 below were taken, with one change: instead of a v0.2 loader, Alex's loader became the standard and our templates moved to his schema. `src/tools/adapter.py` plugs his five tools into the router (`run_any`, `catalog_all`). Negative transactions are now a separate category per the client (2026-10-23): the three money templates still return signed sums and are to be split into amount / adjustments / net.*

**Facts** (sources: section B table `07_crosscheck_<run>.csv`, section A `07_schema_diff_<run>.csv`):
- The three relationship counts in the team are one dataset under three deduplication policies: 59,121 keeps every
  source row (needed for benchmark evidence), 58,911 collapses the 210 exact duplicate participation rows (Alex),
  58,314 was v1's over-collapse by (organisation, activity).
- Where the two tool layers answer the same question, the Cypher template on Aura, the pandas/Decimal computation on the
  CSVs and Alex's published numbers agree (see `agree` column after a live run); the SQL-baseline style of confidence
  now covers the user-facing questions too.
- Alex's tools cannot run on the shared Aura graph today because of typed dates/values, derived labels and the
  `PROVIDES` direction; the gap is a loader change, not a tool change.
- Shared partners between Oxfam GB and WFP are zero on raw identifiers and non-zero only after canonicalisation;
  the number depends entirely on the alias table (7 with ours, 12 with Alex's), so the two tables must be merged and
  reviewed with the pipeline team.

**Decisions to put to the team**
1. One graph, schema v0.2 as proposed in section A; 04 v3 loads it. Alex's loader is retired, his typed/derived
   properties are adopted.
2. One tool package: Alex's four user-facing tools plus the benchmark templates behind one interface that always
   returns `rows, cypher, params, coverage_note` and, where possible, `evidence` (source_id). An adapter maps
   `activity_ids` to `source_id`s for scoring.
3. One resolver: Alex's `resolve_entity` (rapidfuzz + SAME_AS/AGENCY_OF) with our 11 alias groups folded into his tables.
4. Router and abstain/clarify policy stay in notebook 06 and call the merged package; the ReAct agent (notebook 08)
   chains the same tools.